In [1]:
import mysql.connector
import pandas as pd
from mysql.connector import Error

In [6]:
class BaseDatosHR:
    """
    Clase para crear e insertar datos en una Base de Datos relacional MySQL
    de Recursos Humanos - ABC Corporation
    """
    
    def __init__(self, host='localhost', user='root', password='', database='abc_hr', auth_plugin='mysql_native_password'):
        self.host = host
        self.user = user
        self.password = password
        self.database = database
        self.auth_plugin = auth_plugin
        self.conexion = None
        self.cursor = None
    
    def conectar_sin_bd(self):
        """Establece conexión con MySQL sin especificar base de datos"""
        try:
            conexion = mysql.connector.connect(
                host=self.host,
                user=self.user,
                password=self.password,
                auth_plugin=self.auth_plugin
            )
            return conexion
        except Error as e:
            print(f"✗ Error de conexión: {e}")
            return None
    
    def crear_base_de_datos(self):
        """Crea la base de datos si no existe"""
        try:
            conexion = self.conectar_sin_bd()
            if not conexion:
                return False
            
            cursor = conexion.cursor()
            cursor.execute(f"CREATE DATABASE IF NOT EXISTS {self.database}")
            conexion.commit()
            print(f"✓ Base de datos '{self.database}' creada/verificada")
            
            cursor.close()
            conexion.close()
            return True
            
        except Error as e:
            print(f"✗ Error al crear base de datos: {e}")
            return False
    
    def conectar(self):
        """Establece conexión con la base de datos MySQL"""
        try:
            self.conexion = mysql.connector.connect(
                host=self.host,
                user=self.user,
                password=self.password,
                database=self.database,
                auth_plugin=self.auth_plugin
            )
            self.cursor = self.conexion.cursor()
            print(f"✓ Conectado a MySQL: {self.database}")
            return self.conexion
        except Error as e:
            print(f"✗ Error de conexión: {e}")
            return None
    
    def desconectar(self):
        """Cierra la conexión"""
        if self.conexion and self.conexion.is_connected():
            self.cursor.close()
            self.conexion.close()
            print("✓ Desconectado de MySQL")
    
    def crear_tablas(self):
        """Crea la estructura de tablas relacional"""
        try:
            # TABLA 1: DEPARTMENTS (Departamentos)
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS departments (
                    dept_id INT AUTO_INCREMENT PRIMARY KEY,
                    department VARCHAR(100) UNIQUE NOT NULL
                )
            """)
            
            # TABLA 2: JOB_ROLES (Roles de trabajo)
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS job_roles (
                    role_id INT AUTO_INCREMENT PRIMARY KEY,
                    job_role VARCHAR(100) UNIQUE NOT NULL
                )
            """)
            
            # TABLA 3: EMPLOYEE_PERSONAL (Datos Personales Empleado)
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS employee_personal (
                    employee_id INT AUTO_INCREMENT PRIMARY KEY,
                    employee_number INT UNIQUE NOT NULL,
                    age INT,
                    gender VARCHAR(20),
                    marital_status VARCHAR(20),
                    attrition VARCHAR(10),
                    education INT,
                    education_field VARCHAR(100),
                    dept_id INT NOT NULL,
                    role_id INT NOT NULL,
                    
                    FOREIGN KEY (dept_id) REFERENCES departments(dept_id),
                    FOREIGN KEY (role_id) REFERENCES job_roles(role_id)
                )
            """)
            
            # TABLA 4: EMPLOYEE_DETAILS (Datos Trabajo Empleados)
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS employee_details (
                    detail_id INT AUTO_INCREMENT PRIMARY KEY,
                    employee_id INT UNIQUE NOT NULL,
                    distance_from_home INT,
                    environment_satisfaction INT,
                    job_involvement INT,
                    job_level INT,
                    job_satisfaction INT,
                    monthly_income DECIMAL(10, 2),
                    hourly_rate INT,
                    over_time VARCHAR(10),
                    business_travel VARCHAR(50),
                    years_at_company INT,
                    years_in_current_role INT,
                    performance_rating INT,
                    total_working_years INT,
                    training_times_last_year INT,
                    num_companies_worked INT,
                    percent_salary_hike DECIMAL(5, 2),
                    stock_option_level INT,
                    relationship_satisfaction INT,
                    work_life_balance INT,
                    years_since_last_promotion INT,
                    years_with_curr_manager INT,
                    
                    FOREIGN KEY (employee_id) REFERENCES employee_personal(employee_id)
                )
            """)
            
            self.conexion.commit()
            print("✓ Tablas creadas correctamente")
            
        except Error as e:
            print(f"✗ Error al crear tablas: {e}")
            self.conexion.rollback()
    
    def insertar_dimensiones(self, df):
        """Inserta valores únicos en las tablas dimensionales"""
        try:
            # Insertar departamentos únicos
            depts = df['department'].unique()
            for dept in depts:
                self.cursor.execute(
                    "INSERT IGNORE INTO departments (department) VALUES (%s)", (dept,))
                
            
            # Insertar roles únicos
            roles = df['job_role'].unique()
            for role in roles:
                self.cursor.execute(
                    "INSERT IGNORE INTO job_roles (job_role) VALUES (%s)",(role,))
                
                
            self.conexion.commit()
            print(f"✓ Dimensiones sincronizadas")
            
        except Error as e:
            print(f"✗ Error al insertar dimensiones: {e}")
            self.conexion.rollback()
    
    def obtener_diccionario_ids(self, tabla, id_columna, valor_columna):
        """Extrae todos los IDs de una tabla para usarlos en memoria"""
        try:
            self.cursor.execute(f"SELECT {id_columna}, {valor_columna} FROM {tabla}")
            # Crea un diccionario {nombre: id} -> {'Sales': 1, 'Research': 2}
            return {row[1]: row[0] for row in self.cursor.fetchall()}
        except Error as e:
            print(f"✗ Error al cargar diccionario de {tabla}: {e}")
            return {}
    
    def insertar_empleados(self, df):
        """Inserta todos los empleados garantizando integridad y alto rendimiento"""
        try:
            # 1. Pre-cargar mapeo de IDs en memoria (0 consultas en el bucle)
            mapa_dept = self.obtener_diccionario_ids('departments', 'dept_id', 'department')
            mapa_roles = self.obtener_diccionario_ids('job_roles', 'role_id', 'job_role')

            contador = 0
            errores = 0
            
            for idx, row in df.iterrows():
                try:
                    # Buscar IDs en el diccionario de Python (O(1) en memoria, hiper-rápido)
                    dept_id = mapa_dept.get(row['department'])
                    role_id = mapa_roles.get(row['job_role'])

                    if not dept_id or not role_id:
                        raise ValueError(f"Falta departamento o rol para empleado {row['employee_number']}")

                    # Iniciar transacción para ESTE empleado
                    self.conexion.start_transaction()
                    
                    # Insertar datos personales
                    self.cursor.execute("""
                        INSERT INTO employee_personal (
                            employee_number, age, gender, marital_status, attrition,
                            education, education_field, dept_id, role_id
                        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                    """, (
                        int(row['employee_number']),
                        int(row['age']) if pd.notna(row['age']) else None,
                        row['gender'],
                        row['marital_status'],
                        row['attrition'],
                        int(row['education']) if pd.notna(row['education']) else None,
                        row['education_field'],
                        dept_id,
                        role_id
                    ))
                    
                    # Obtener el ID del empleado insertado
                    employee_id = self.cursor.lastrowid
                    
                    # Insertar datos de trabajo
                    self.cursor.execute("""
                        INSERT INTO employee_details (
                            employee_id, distance_from_home, environment_satisfaction,
                            job_involvement, job_level, job_satisfaction, monthly_income,
                            hourly_rate, over_time, business_travel, years_at_company,
                            years_in_current_role, performance_rating, total_working_years,
                            training_times_last_year, num_companies_worked, percent_salary_hike,
                            stock_option_level, relationship_satisfaction, work_life_balance,
                            years_since_last_promotion, years_with_curr_manager
                        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    """, (
                        employee_id,
                        int(row['distance_from_home']) if pd.notna(row['distance_from_home']) else None,
                        int(row['environment_satisfaction']) if pd.notna(row['environment_satisfaction']) else None,
                        int(row['job_involvement']) if pd.notna(row['job_involvement']) else None,
                        int(row['job_level']) if pd.notna(row['job_level']) else None,
                        int(row['job_satisfaction']) if pd.notna(row['job_satisfaction']) else None,
                        float(row['monthly_income']) if pd.notna(row['monthly_income']) else None,
                        int(row['hourly_rate']) if pd.notna(row['hourly_rate']) else None,
                        row['over_time'],
                        row['business_travel'],
                        int(row['years_at_company']) if pd.notna(row['years_at_company']) else None,
                        int(row['years_in_current_role']) if pd.notna(row['years_in_current_role']) else None,
                        int(row.get('performance_rating')) if pd.notna(row.get('performance_rating')) else None,
                        int(row.get('total_working_years')) if pd.notna(row.get('total_working_years')) else None,
                        int(row.get('training_times_last_year')) if pd.notna(row.get('training_times_last_year')) else None,
                        int(row.get('num_companies_worked')) if pd.notna(row.get('num_companies_worked')) else None,
                        float(row.get('percent_salary_hike')) if pd.notna(row.get('percent_salary_hike')) else None,
                        int(row.get('stock_option_level')) if pd.notna(row.get('stock_option_level')) else None,
                        int(row.get('relationship_satisfaction')) if pd.notna(row.get('relationship_satisfaction')) else None,
                        int(row.get('work_life_balance')) if pd.notna(row.get('work_life_balance')) else None,
                        int(row.get('years_since_last_promotion')) if pd.notna(row.get('years_since_last_promotion')) else None,
                        int(row.get('years_with_curr_manager')) if pd.notna(row.get('years_with_curr_manager')) else None
                    ))

                    # 2. COMMIT ATÓMICO: Solo guardamos si AMBAS tablas se insertaron correctamente
                    self.conexion.commit()
                    contador += 1
                    
                except Exception as e:
                    # 3. ROLLBACK DE FILA: Si una tabla falla, se cancela la transacción de ESTE empleado
                    self.conexion.rollback()
                    errores += 1
                    print(f"Error en empleado {row.get('employee_number', 'Desconocido')}: {e}")
                    continue
            
            print(f"✓ {contador} empleados insertados ({errores} errores)")
            
        except Error as e:
            print(f"✗ Error al insertar empleados: {e}")
            self.conexion.rollback()
    
    def ejecutar_flujo_completo(self, ruta_csv):
        """Ejecuta todo el flujo de creación e inserción de datos"""
        try:
            print("\n" + "="*80)
            print("CREACIÓN DE BASE DE DATOS HR - ABC CORPORATION")
            print("="*80 + "\n")
            
            # 1. Crear base de datos
            if not self.crear_base_de_datos():
                return
            
            # 2. Conectar
            if not self.conectar():
                return
            
            # 3. Crear tablas
            self.crear_tablas()
            
            # 4. Cargar datos del CSV
            print(f"\nCargando datos desde: {ruta_csv}")
            df = pd.read_csv(ruta_csv)
            print(f"✓ {len(df)} registros cargados")
            
            # 5. Insertar dimensiones
            print("\nInsertando datos dimensionales...")
            self.insertar_dimensiones(df)
            
            # 6. Insertar empleados
            print("\nInsertando datos de empleados...")
            self.insertar_empleados(df)
            
            print("\n✓ PROCESO COMPLETADO EXITOSAMENTE\n")
            
        except Exception as e:
            print(f"✗ Error en el flujo: {e}")
        finally:
            self.desconectar()

In [7]:
# ==============================================================================
# EJECUCIÓN
# ==============================================================================

if __name__ == "__main__":
    pwd = input("Introduce el password de MySQL: ")
    
    # Crear instancia de la base de datos con credenciales de MySQL
    bd = BaseDatosHR(
        host='localhost',
        user='root',
        password=pwd,  # Input de tu contraseña
        database='abc_hr',
        auth_plugin='mysql_native_password'
    )
    
    # Ejecutar flujo completo
    bd.ejecutar_flujo_completo("../df_hr_limpio.csv")


CREACIÓN DE BASE DE DATOS HR - ABC CORPORATION

✓ Base de datos 'abc_hr' creada/verificada
✓ Conectado a MySQL: abc_hr
✓ Tablas creadas correctamente

Cargando datos desde: ../df_hr_limpio.csv
✓ 1470 registros cargados

Insertando datos dimensionales...
✓ Dimensiones sincronizadas

Insertando datos de empleados...
Error en empleado 1: Transaction already in progress
✓ 1469 empleados insertados (1 errores)

✓ PROCESO COMPLETADO EXITOSAMENTE

✓ Desconectado de MySQL
